By: Jesse Anderson

Implementation of water depth / bathymetry algorithm as described here: https://www.mdpi.com/2072-4292/13/8/14691024

Code translated from GEE code here: https://github.com/CoralMapping/GEE_Sentinel2_Bathymetry_Paper

In [7]:
import rioxarray as rx
import xarray as xr
import xrspatial as xs

In [8]:
def mask(xarr):
    # Initial masking is done by QA60 band. I think that is GEE specific

    green = xarr.B03
    water_vapor = xarr.B09

    ma = xarr.where(
        ~xarr.SCL.isin([3, 4, 5, 8, 9, 10])
        & (water_vapor < 300)
        & (water_vapor > 50)
        & (green > 100),
    )
    kernel = xs.convolution.circle_kernel(1, 1, 1)
    ma = xarr.map(xs.focal.focal_stats, kernel=kernel, stats_funcs=["min"]).isel(
        stats=0
    )

    xarr = xarr.where(ma)

    # Maybe these should be done above, but this is how it is in the official code
    red_edge = xarr.B05
    nir = xarr.B08

    xarr = xarr.where(red_edge < 1000).where(nir < 300)

    # This is really ndwi, because of the bands
    ndwi = xs.multispectral.ndvi(green, nir)

    return xarr.where(ndwi > 0).to_array(name="band")

In [9]:
from math import exp
import numpy as np


def calculate_depth(xarr):
    bigrrs = xarr / 31415.926
    rrsvec = bigrrs / (bigrrs * 1.7 + 0.52)
    rrsvec1k = rrsvec * 1000

    # This is the chlorophyll-a level, and may be too high for a lot of locations.
    # It's worth trying different values to see how results change
    chla = 0.5
    m0 = 52.073 * exp(0.957 * chla)
    m1 = 50.156 * exp(0.957 * chla)

    lnrrsvec = np.log(rrsvec1k)
    blue = lnrrsvec.B02
    green = lnrrsvec.B03
    depth = (blue / green) * m0 - m1

    # This clamping is in the original paper. It's worth going without, or leaving
    # extreme values as nan to see where estimates are saturated at high or low values.
    return depth.where(depth > 0, 0).where(depth < 20, 20).where(~xarr.B02.isnull())

In [10]:
import geopandas as gpd
import planetary_computer
import pystac_client
from shapely.geometry import box

# bounds for Samoa
bounds = [-172.8, -13.8, -172.1, -13.37]
aoi = box(*bounds)

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=aoi,
    # be aware of offset if loading recent data
    datetime="2019",
    query={"eo:cloud_cover": {"lt": 10}},
)

In [11]:
import odc.stac

ds = odc.stac.load(
    search.items(),
    chunks=dict(x=2048, y=2048),
    bbox=bounds,
    crs=3832,
    bands=["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B09", "SCL"],
)

In [ ]:
estimate = calculate_depth(
    ds.median(dim="time")
).to_dataset(name="depth").compute()

estimate

In [ ]:
estimate.depth.odc.explore(cmap="Blues")